# BRISC2025 Brain MRI Multi-Task Learning

This self-contained notebook trains one shared encoder with separate four-class classification and tumor-segmentation heads. The official test set is used only after validation-based model selection and threshold selection. The raw dataset and existing split CSVs are read-only inputs.

## 1. Environment setup

Uses the supported torchvision ConvNeXt-Tiny and Swin-Tiny implementations with their official ImageNet weight enums. This keeps architecture construction and checkpoint loading reproducible without an external `src/` package or an implicit timm API/version switch.

In [ ]:
import os, sys, time, json, math, random, hashlib, platform, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageDraw, ImageOps
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision.models import (convnext_tiny, swin_t,
    ConvNeXt_Tiny_Weights, Swin_T_Weights)
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix, roc_auc_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print({"GPU": torch.cuda.get_device_name(0), "GPU_VRAM_GB": round(props.total_memory/2**30, 2),
           "CUDA": torch.version.cuda, "torch": torch.__version__, "torchvision": torchvision.__version__,
           "timm": "not required (torchvision model API)", "free_GPU_memory_GB": round(torch.cuda.mem_get_info()[0]/2**30, 2)})
else:
    print({"GPU": None, "GPU_VRAM_GB": 0, "CUDA": torch.version.cuda, "torch": torch.__version__,
           "torchvision": torchvision.__version__, "timm": "not required (torchvision model API)", "device": "CPU fallback"})


## 2. Configuration

For Colab, mount Drive or upload the dataset, then set `DATA_ROOT`, `SPLIT_DIR`, and `OUTPUT_ROOT` in this cell. Keep the existing CSVs unchanged. Horizonal/vertical flips are disabled: view orientation is mixed and flipping could alter anatomy/orientation. Training uses only mild image brightness/contrast variation; masks receive the identical resize and no intensity transforms.

In [ ]:
# Portable local defaults; environment variables can point at a mounted dataset or output folder.
def find_project_root():
    configured = os.environ.get("NEUROSCAN_PROJECT_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if (root / "train_brain_tumor_multitask.ipynb").is_file():
            return root
        raise FileNotFoundError(f"NEUROSCAN_PROJECT_ROOT does not contain the notebook: {root}")
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "train_brain_tumor_multitask.ipynb").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Open this notebook from the NeuroScan repository, or set NEUROSCAN_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = Path(os.environ.get("NEUROSCAN_DATA_ROOT", PROJECT_ROOT / "data/raw/brisc2025")).expanduser().resolve()
SPLIT_DIR = Path(os.environ.get("NEUROSCAN_SPLIT_DIR", PROJECT_ROOT / "data/splits")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("NEUROSCAN_OUTPUT_ROOT", PROJECT_ROOT)).expanduser().resolve()
CHECKPOINT_DIR = OUTPUT_ROOT / "artifacts/checkpoints"
PLOT_DIR = OUTPUT_ROOT / "artifacts/plots"
REPORT_DIR = OUTPUT_ROOT / "artifacts/reports"
OUTPUT_DIR = OUTPUT_ROOT / "outputs"
for _p in (CHECKPOINT_DIR, PLOT_DIR, REPORT_DIR, OUTPUT_DIR): _p.mkdir(parents=True, exist_ok=True)
CLASSES = ("glioma", "meningioma", "pituitary", "non_tumorous")
FOLDER_LABELS = {"glioma":"glioma", "meningioma":"meningioma", "pituitary":"pituitary", "no_tumor":"non_tumorous", "non_tumorous":"non_tumorous"}
CLASS_TO_IDX = {name:i for i,name in enumerate(CLASSES)}
IMAGE_SIZE = 224
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 0  # Safe default for Colab/Jupyter multiprocessing; increase after validating the runtime
EPOCHS = 20
PATIENCE = 5
SEED = 42
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
LAMBDA_CLASSIFICATION = 1.0
LAMBDA_SEGMENTATION = 1.0
LABEL_SMOOTHING = 0.02
GLIOMA_RECALL_TARGET = 0.95
MASK_THRESHOLD_GRID = np.linspace(0.1, 0.9, 17)
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
assert (DATA_ROOT/"classification_task").is_dir() and (DATA_ROOT/"segmentation_task").is_dir()
assert all((SPLIT_DIR/f).is_file() for f in ("classification_train.csv","classification_validation.csv","segmentation_train.csv","segmentation_validation.csv"))


## 3–5. Dataset inspection, split/leakage audit, and image-mask pairs

The existing splits are loaded read-only. The project audit found 7 exact-content raw train/test duplicate groups; inspection confirmed those examples are absent from the generated training and validation CSVs. The split audit is repeated below and halts if it finds leakage. Masks contain interpolation edge values, so this binary-mask task thresholds grayscale at 127; raw masks are never changed.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

def resolve_data_path(value):
    path=Path(str(value))
    if path.is_file(): return path
    parts=path.parts
    for task in ("classification_task","segmentation_task"):
        if task in parts:
            candidate=DATA_ROOT.joinpath(*parts[parts.index(task):])
            if candidate.is_file(): return candidate
    raise FileNotFoundError(f"Split path cannot be resolved under DATA_ROOT={DATA_ROOT}: {value}")

def mask_class_from_name(path):
    s=Path(path).stem.lower()
    for code,label in (("_gl_","glioma"),("_me_","meningioma"),("_pi_","pituitary")):
        if code in s: return label
    raise ValueError(f"Unrecognized BRISC segmentation class code: {path}")

def audit_raw_dataset():
    counts={}; bad=[]; empty_masks=[]; dims=Counter(); mask_hist=np.zeros(256,dtype=np.int64); pair_rows={}
    for split in ("train","test"):
        for folder in sorted((DATA_ROOT/"classification_task"/split).iterdir()):
            if folder.is_dir(): counts[(split,FOLDER_LABELS.get(folder.name,folder.name))]=len(list(folder.glob("*.jpg")))
        ib=DATA_ROOT/"segmentation_task"/split/"images"; mb=DATA_ROOT/"segmentation_task"/split/"masks"
        imgs={p.stem:p for p in ib.glob("*.jpg")}; masks={p.stem:p for p in mb.glob("*.png")}
        if imgs.keys()!=masks.keys(): raise ValueError(f"Missing pairs for {split}: images-only={len(imgs.keys()-masks.keys())}, masks-only={len(masks.keys()-imgs.keys())}")
        pair_rows[split]=[{"image":str(imgs[k]),"mask":str(masks[k]),"label":mask_class_from_name(imgs[k])} for k in sorted(imgs)]
        for k in imgs:
            try:
                with Image.open(imgs[k]) as im, Image.open(masks[k]) as ma:
                    if im.size!=ma.size: bad.append((str(imgs[k]),"dimension mismatch"))
                    dims[im.size]+=1; mh=np.asarray(ma.convert("L").histogram(),dtype=np.int64); mask_hist += mh
                    if int(mh[1:].sum())==0: empty_masks.append(str(masks[k]))
            except Exception as e: bad.append((str(imgs[k]),str(e)))
    raw_duplicate_groups={}
    for task,col in (("classification","path"),("segmentation","image")):
        if task=="classification":
            raw_train=sorted((DATA_ROOT/task/"train").rglob("*.jpg"));raw_test=sorted((DATA_ROOT/task/"test").rglob("*.jpg"))
        else:
            raw_train=sorted((DATA_ROOT/task/"train"/"images").glob("*.jpg"));raw_test=sorted((DATA_ROOT/task/"test"/"images").glob("*.jpg"))
        train_by_hash={}
        for p in raw_train: train_by_hash.setdefault(sha256(p),[]).append(str(p))
        test_hashes={sha256(p) for p in raw_test}
        raw_duplicate_groups[task]=[v for h,v in train_by_hash.items() if h in test_hashes]
        frame_paths=[]
        for split in ("train","validation"):
            frame_paths.extend(str(resolve_data_path(p)) for p in pd.read_csv(SPLIT_DIR/f"{task}_{split}.csv")[col].tolist())
        included={sha256(p) for p in frame_paths}
        if included & test_hashes: raise ValueError(f"{task}: official test content appears in an existing training/validation split")
        print(task,"raw train/test exact duplicate groups:",len(raw_duplicate_groups[task]),"; all excluded from supplied splits")
    class_records={}
    for split in ("train","test"):
        base=DATA_ROOT/"classification_task"/split
        class_records[split]=[{"path":str(p),"label":FOLDER_LABELS[p.parent.name]} for c in sorted(base.iterdir()) if c.is_dir() for p in sorted(c.glob("*.jpg"))]
        for r in class_records[split]:
            try:
                with Image.open(r["path"]) as im: im.verify()
            except Exception as e: bad.append((r["path"],str(e)))
    # Confirm filename-derived segmentation labels against class folders wherever basenames overlap.
    for split in ("train","test"):
        class_label={Path(r["path"]).stem.lower():r["label"] for r in class_records[split]}
        mismatches=[]
        for rec in pair_rows[split]:
            stem=Path(rec["image"]).stem.lower()
            if stem in class_label and class_label[stem]!=rec["label"]: mismatches.append((stem,class_label[stem],rec["label"]))
        if mismatches: raise ValueError(f"{split} image/mask class-code mismatch: {mismatches[:5]}")
        print(split,"segmentation labels matched to classification folders for",sum(Path(r["image"]).stem.lower() in class_label for r in pair_rows[split]),"pairs")
    for task,rows,key in (("classification",class_records["train"],"path"),("segmentation",pair_rows["train"],"image")):
        val=pd.read_csv(SPLIT_DIR/f"{task}_validation.csv"); train=pd.read_csv(SPLIT_DIR/f"{task}_train.csv")
        col="path" if task=="classification" else "image"
        for frame in (train,val):
            frame[col]=[str(resolve_data_path(p)) for p in frame[col]]
            missing=[p for p in frame[col] if not Path(p).is_file()]
            if missing: raise FileNotFoundError(f"{task} split references absent files, e.g. {missing[:3]}")
        tr_hash={sha256(p) for p in train[col]}; va_hash={sha256(p) for p in val[col]}
        testdir=DATA_ROOT/task/"test"; testpaths=list(testdir.rglob("*.jpg"))
        te_hash={sha256(p) for p in testpaths}
        if tr_hash&va_hash or tr_hash&te_hash or va_hash&te_hash: raise ValueError(f"{task} exact-content leakage detected")
        print(task,"split sizes",len(train),len(val),"exact leakage: none")
    cls_assign={}; seg_assign={}
    for task,col,assignment in (("classification","path",cls_assign),("segmentation","image",seg_assign)):
        for split in ("train","validation"):
            d=pd.read_csv(SPLIT_DIR/f"{task}_{split}.csv")
            assignment.update({sha256(resolve_data_path(p)):split for p in d[col]})
    mismatch=[h for h in cls_assign.keys()&seg_assign.keys() if cls_assign[h]!=seg_assign[h]]
    if mismatch: raise ValueError(f"Cross-task split mismatch: {len(mismatch)} exact duplicate images")
    if bad: raise ValueError(f"Corrupt or dimension-invalid files: {bad[:5]}")
    uniq=np.flatnonzero(mask_hist)
    summary={"classification_counts":{f"{s}/{c}":n for (s,c),n in counts.items()},"segmentation_pairs":{k:len(v) for k,v in pair_rows.items()},"dimensions":{str(k):v for k,v in dims.items()},"mask_unique_values":uniq.tolist(),"mask_histogram":{int(i):int(mask_hist[i]) for i in uniq},"negative_masks":bool(sum(mask_hist[1:])==0),"empty_masks":empty_masks,"raw_train_test_duplicate_groups":raw_duplicate_groups}
    print(json.dumps(summary,indent=2)); return class_records,pair_rows,summary
CLASS_RECORDS, SEG_RECORDS, DATA_AUDIT = audit_raw_dataset()


## 6. Dataset and DataLoader

Classification and segmentation batches are drawn from their existing task splits and passed through the same encoder during each optimization step. The paired image basename contains a documented source code (`gl`, `me`, `pi`); the current CSV segmentation label column is `unknown`, so labels are parsed from that verified convention. Masks are binarized in memory with `>127` and resized with nearest-neighbor interpolation.

In [ ]:
class MRIRecordDataset(Dataset):
    def __init__(self, rows, task, train=False, size=IMAGE_SIZE):
        self.rows=rows.to_dict("records") if isinstance(rows,pd.DataFrame) else list(rows)
        self.task=task; self.train=train; self.size=size
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        if self.task=="classification":
            with Image.open(r["path"]) as im: im=ImageOps.exif_transpose(im).convert("RGB").resize((self.size,self.size),Image.Resampling.BILINEAR)
            if self.train:
                if random.random()<.5: im=ImageEnhance.Brightness(im).enhance(random.uniform(.92,1.08))
                if random.random()<.5: im=ImageEnhance.Contrast(im).enhance(random.uniform(.92,1.08))
            x=torch.from_numpy(np.asarray(im,dtype=np.float32).copy()).permute(2,0,1)/255
            x=(x-torch.tensor(MEAN)[:,None,None])/torch.tensor(STD)[:,None,None]
            return x,CLASS_TO_IDX[FOLDER_LABELS.get(r["label"],r["label"])],r["path"]
        with Image.open(r["image"]) as im: im=ImageOps.exif_transpose(im).convert("RGB").resize((self.size,self.size),Image.Resampling.BILINEAR)
        with Image.open(r["mask"]) as m: m=m.convert("L").resize((self.size,self.size),Image.Resampling.NEAREST)
        if self.train and random.random()<.5:
            im=ImageEnhance.Contrast(im).enhance(random.uniform(.92,1.08))
        x=torch.from_numpy(np.asarray(im,dtype=np.float32).copy()).permute(2,0,1)/255
        x=(x-torch.tensor(MEAN)[:,None,None])/torch.tensor(STD)[:,None,None]
        y=(torch.from_numpy(np.asarray(m,dtype=np.uint8).copy()).unsqueeze(0)>127).float()
        return x,y,r["image"],r["mask"]

def get_split(task,split):
    df=pd.read_csv(SPLIT_DIR/f"{task}_{split}.csv")
    if task=="segmentation":
        df["image"]=[str(resolve_data_path(p)) for p in df["image"]]
        df["mask"]=[str(resolve_data_path(p)) for p in df["mask"]]
        df["label"]=[mask_class_from_name(p) for p in df["image"]]
        # The persisted split files remain untouched; all source records are from official train.
        for p in df.image:
            if "/train/" not in str(p): raise ValueError("Validation segmentation split must derive from official train folder")
    else:
        df["path"]=[str(resolve_data_path(p)) for p in df.path]
        df["label"]=[FOLDER_LABELS.get(Path(p).parent.name,Path(p).parent.name) for p in df.path]
    return df

CLS_TRAIN=get_split("classification","train"); CLS_VAL=get_split("classification","validation")
SEG_TRAIN=get_split("segmentation","train"); SEG_VAL=get_split("segmentation","validation")
train_loaders={"classification":DataLoader(MRIRecordDataset(CLS_TRAIN,"classification",True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda"),
               "segmentation":DataLoader(MRIRecordDataset(SEG_TRAIN,"segmentation",True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda")}
val_loaders={"classification":DataLoader(MRIRecordDataset(CLS_VAL,"classification"),batch_size=BATCH_SIZE,num_workers=NUM_WORKERS),
             "segmentation":DataLoader(MRIRecordDataset(SEG_VAL,"segmentation"),batch_size=BATCH_SIZE,num_workers=NUM_WORKERS)}


## 7. Shared model definitions

Both candidates use one shared hierarchical encoder (the same module produces classification features and decoder skips) and independent classification/segmentation heads. Torchvision 0.29.1 was inspected: ConvNeXt stages emit NCHW at 56/28/14/7; Swin stages emit NHWC at the same resolutions. The model converts Swin maps to NCHW before decoding. No silent architecture fallback is used.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,cin,cout):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(cin,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout),nn.GELU(),nn.Conv2d(cout,cout,3,padding=1,bias=False),nn.BatchNorm2d(cout),nn.GELU())
    def forward(self,x): return self.net(x)

class MultiTaskModel(nn.Module):
    STAGE_IDS=(1,3,5,7); DIMS=(96,192,384,768)
    def __init__(self,architecture,pretrained=True,num_classes=4):
        super().__init__(); self.architecture=architecture
        if architecture=="convnext_tiny":
            weights=ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None; backbone=convnext_tiny(weights=weights)
        elif architecture=="swin_tiny":
            weights=Swin_T_Weights.DEFAULT if pretrained else None; backbone=swin_t(weights=weights)
        else: raise ValueError(f"Unsupported architecture {architecture!r}; choose convnext_tiny or swin_tiny")
        self.encoder=backbone.features
        self.classifier=nn.Sequential(nn.LayerNorm(768),nn.Linear(768,num_classes))
        self.proj=nn.ModuleList([nn.Conv2d(96,96,1),nn.Conv2d(192,128,1),nn.Conv2d(384,128,1),nn.Conv2d(768,128,1)])
        self.dec3=ConvBlock(256,128); self.dec2=ConvBlock(256,96); self.dec1=ConvBlock(192,64)
        self.mask_head=nn.Conv2d(64,1,1)
    def forward(self,x):
        skips=[]
        for i,layer in enumerate(self.encoder):
            x=layer(x)
            if i in self.STAGE_IDS:
                feat=x.permute(0,3,1,2).contiguous() if self.architecture=="swin_tiny" else x
                skips.append(feat)
        f1,f2,f3,f4=skips
        cls=self.classifier(F.adaptive_avg_pool2d(f4,1).flatten(1))
        d=F.interpolate(self.proj[3](f4),size=f3.shape[-2:],mode="bilinear",align_corners=False);d=self.dec3(torch.cat([d,self.proj[2](f3)],1))
        d=F.interpolate(d,size=f2.shape[-2:],mode="bilinear",align_corners=False);d=self.dec2(torch.cat([d,self.proj[1](f2)],1))
        d=F.interpolate(d,size=f1.shape[-2:],mode="bilinear",align_corners=False);d=self.dec1(torch.cat([d,self.proj[0](f1)],1))
        seg=F.interpolate(self.mask_head(d),size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)
        return {"classification":cls,"segmentation":seg}

def parameter_count(model): return sum(p.numel() for p in model.parameters())


## 8–9. Losses and training utilities

The objective is `lambda_cls * cross_entropy + lambda_seg * (BCE + soft Dice)`. Equal initial task weights are configurable. Epoch logs retain each task loss, so a persistent scale imbalance is visible. Model selection uses the mean of validation classification macro F1 and mean per-image segmentation Dice; neither task alone determines the best checkpoint.

In [ ]:
def dice_loss(logits,target,eps=1e-6):
    p=torch.sigmoid(logits); dims=(1,2,3); inter=(p*target).sum(dims); den=p.sum(dims)+target.sum(dims)
    return 1-((2*inter+eps)/(den+eps)).mean()

def segmentation_loss(logits,target): return .5*F.binary_cross_entropy_with_logits(logits,target)+.5*dice_loss(logits,target)
def combined_loss(cls_logits,cls_y,seg_logits,seg_y):
    lc=F.cross_entropy(cls_logits,cls_y,label_smoothing=LABEL_SMOOTHING)
    ls=segmentation_loss(seg_logits,seg_y)
    return LAMBDA_CLASSIFICATION*lc+LAMBDA_SEGMENTATION*ls,lc.detach(),ls.detach()

def cls_metrics(y,p):
    pred=p.argmax(1); pr,re,f1,_=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)
    cm=confusion_matrix(y,pred,labels=range(4))
    return {"accuracy":float(accuracy_score(y,pred)),"balanced_accuracy":float(balanced_accuracy_score(y,pred)),"macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),"per_class_precision":pr.tolist(),"per_class_recall":re.tolist(),"per_class_f1":f1.tolist(),"glioma_recall":float(re[0]),"glioma_meningioma_confusion":int(cm[0,1]),"confusion_matrix":cm.tolist()}

def seg_metrics(prob,target,threshold=.5):
    pred=prob>=threshold; truth=target>=.5; axes=(1,2,3)
    inter=(pred&truth).sum(axes); den=pred.sum(axes)+truth.sum(axes); union=(pred|truth).sum(axes)
    dice=np.where(den==0,1.,2*inter/np.maximum(den,1)); iou=np.where(union==0,1.,inter/np.maximum(union,1))
    tp=(pred&truth).sum(); fp=(pred&~truth).sum(); fn=(~pred&truth).sum()
    return {"dice_per_image":dice,"iou_per_image":iou,"dice_mean":float(dice.mean()),"iou_mean":float(iou.mean()),"pixel_precision":float(tp/max(1,tp+fp)),"pixel_recall":float(tp/max(1,tp+fn))}

@torch.no_grad()
def validate(model,loader_cls,loader_seg):
    model.eval(); ys=[];ps=[]; seg_probs=[];seg_truth=[]; losses=[];seg_losses=[]
    for x,y,_ in loader_cls:
        x=x.to(DEVICE);y=y.to(DEVICE);out=model(x);losses.append(float(F.cross_entropy(out["classification"],y)));ys.extend(y.cpu().tolist());ps.extend(torch.softmax(out["classification"],1).cpu().numpy())
    for x,y,_,_ in loader_seg:
        out=model(x.to(DEVICE));seg_losses.append(float(segmentation_loss(out["segmentation"],y.to(DEVICE))));seg_probs.extend(torch.sigmoid(out["segmentation"]).cpu().numpy());seg_truth.extend(y.numpy())
    y=np.asarray(ys);p=np.asarray(ps);sp=np.asarray(seg_probs);sy=np.asarray(seg_truth);cm=cls_metrics(y,p);sm=seg_metrics(sp,sy)
    return {"classification":cm,"segmentation":{k:v for k,v in sm.items() if k not in ("dice_per_image","iou_per_image")},"classification_val_loss":float(np.mean(losses)),"segmentation_val_loss":float(np.mean(seg_losses)) if seg_losses else None},y,p,sp,sy

def save_checkpoint(path,model,optimizer,scaler,epoch,metadata,best_score):
    torch.save({"state_dict":model.state_dict(),"optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),"epoch":epoch,"metadata":metadata,"best_score":best_score},path)

def train_candidate(architecture,resume=True):
    seed_everything(); model=MultiTaskModel(architecture,pretrained=True).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",patience=2,factor=.5)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    start=0; best=-1.; wait=0; hist=[]; latest=CHECKPOINT_DIR/f"latest_{architecture}.pth"; bestpath=CHECKPOINT_DIR/f"candidate_{architecture}_best.pth"
    if resume and latest.exists():
        ck=torch.load(latest,map_location=DEVICE,weights_only=False)
        if ck.get("metadata",{}).get("architecture")!=architecture: raise ValueError("Resume checkpoint architecture mismatch")
        model.load_state_dict(ck["state_dict"]);optimizer.load_state_dict(ck["optimizer"]);scaler.load_state_dict(ck["scaler"]);start=int(ck["epoch"]);best=float(ck["best_score"])
        hp=OUTPUT_DIR/f"history_{architecture}.json"
        if hp.exists():hist=json.loads(hp.read_text())
        print("Resuming",architecture,"at epoch",start)
    nsteps=max(len(train_loaders["classification"]),len(train_loaders["segmentation"]))
    started=time.time()
    for epoch in range(start,EPOCHS):
        model.train(); itc=iter(train_loaders["classification"]);its=iter(train_loaders["segmentation"]); sums=np.zeros(3); train_cls_y=[];train_cls_prob=[];train_seg_dice=[];train_seg_iou=[]; optimizer.zero_grad(set_to_none=True)
        for step in range(nsteps):
            try: cx,cy,_=next(itc)
            except StopIteration: itc=iter(train_loaders["classification"]);cx,cy,_=next(itc)
            try: sx,sy,_,_=next(its)
            except StopIteration: its=iter(train_loaders["segmentation"]);sx,sy,_,_=next(its)
            cx,cy,sx,sy=cx.to(DEVICE),cy.to(DEVICE),sx.to(DEVICE),sy.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type,enabled=DEVICE.type=="cuda"):
                co=model(cx);so=model(sx);total,lc,ls=combined_loss(co["classification"],cy,so["segmentation"],sy);scaled=total/GRAD_ACCUM_STEPS
            scaler.scale(scaled).backward();sums += [total.item(),lc.item(),ls.item()]
            train_cls_y.extend(cy.detach().cpu().tolist());train_cls_prob.extend(torch.softmax(co["classification"].detach(),1).cpu().numpy())
            seg_batch=seg_metrics(torch.sigmoid(so["segmentation"].detach()).cpu().numpy(),sy.detach().cpu().numpy());train_seg_dice.extend(seg_batch["dice_per_image"].tolist());train_seg_iou.extend(seg_batch["iou_per_image"].tolist())
            if (step+1)%GRAD_ACCUM_STEPS==0 or step+1==nsteps:
                scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
        metrics,yv,pv,sv,gtv=validate(model,val_loaders["classification"],val_loaders["segmentation"])
        joint=(metrics["classification"]["macro_f1"]+metrics["segmentation"]["dice_mean"])/2;scheduler.step(joint)
        train_cls_metrics=cls_metrics(np.asarray(train_cls_y),np.asarray(train_cls_prob));row={"architecture":architecture,"epoch":epoch+1,"train_total_loss":float(sums[0]/nsteps),"train_classification_loss":float(sums[1]/nsteps),"train_segmentation_loss":float(sums[2]/nsteps),"train_accuracy":train_cls_metrics["accuracy"],"train_balanced_accuracy":train_cls_metrics["balanced_accuracy"],"train_macro_f1":train_cls_metrics["macro_f1"],"train_glioma_recall":train_cls_metrics["glioma_recall"],"train_segmentation_dice":float(np.mean(train_seg_dice)),"train_segmentation_iou":float(np.mean(train_seg_iou)),"validation_classification_loss":metrics["classification_val_loss"],"validation_segmentation_loss":metrics["segmentation_val_loss"],"validation_joint_score":joint,"validation_macro_f1":metrics["classification"]["macro_f1"],"validation_glioma_recall":metrics["classification"]["glioma_recall"],"validation_balanced_accuracy":metrics["classification"]["balanced_accuracy"],"validation_dice":metrics["segmentation"]["dice_mean"],"validation_iou":metrics["segmentation"]["iou_mean"],"validation_metrics":metrics}
        for ci,cname in enumerate(CLASSES):
            row[f"train_{cname}_precision"]=train_cls_metrics["per_class_precision"][ci];row[f"train_{cname}_recall"]=train_cls_metrics["per_class_recall"][ci];row[f"train_{cname}_f1"]=train_cls_metrics["per_class_f1"][ci]
            row[f"validation_{cname}_precision"]=metrics["classification"]["per_class_precision"][ci];row[f"validation_{cname}_recall"]=metrics["classification"]["per_class_recall"][ci];row[f"validation_{cname}_f1"]=metrics["classification"]["per_class_f1"][ci]
        hist.append(row);pd.DataFrame([{k:v for k,v in r.items() if not isinstance(v,dict)} for r in hist]).to_csv(OUTPUT_DIR/"training_history.csv",index=False)
        metadata={"architecture":architecture,"classes":CLASSES,"input_size":IMAGE_SIZE,"normalization":{"mean":MEAN,"std":STD},"grayscale_to_rgb":True,"mask_threshold":.5,"threshold":None,"task_loss_weights":{"classification":LAMBDA_CLASSIFICATION,"segmentation":LAMBDA_SEGMENTATION},"seed":SEED,"dataset_root":str(DATA_ROOT),"split_files":[str(SPLIT_DIR/f) for f in ("classification_train.csv","classification_validation.csv","segmentation_train.csv","segmentation_validation.csv")],"validation_metrics":metrics,"selection_criterion":"mean(validation classification macro-F1, validation per-image mean Dice)"}
        save_checkpoint(latest,model,optimizer,scaler,epoch+1,metadata,best)
        if joint>best:
            best=joint;wait=0;save_checkpoint(bestpath,model,optimizer,scaler,epoch+1,metadata,best)
        else:wait+=1
        (OUTPUT_DIR/f"history_{architecture}.json").write_text(json.dumps(hist,indent=2));print(architecture,row)
        if wait>=PATIENCE:break
    return {"architecture":architecture,"parameters":parameter_count(model),"training_seconds":time.time()-started,"best_epoch":max(hist,key=lambda r:r["validation_joint_score"])["epoch"],"best_checkpoint":str(bestpath),"history":hist}


## 10–11. Train and compare candidates

Run both candidates with the same official train/validation CSVs and criterion. Pretrained weights are requested explicitly; a download/cache failure stops with an error instead of silently changing initialization. Set `EPOCHS`, batch size, and accumulation above to fit the selected Colab GPU.

In [ ]:
RUN_TRAINING = True
candidate_results=[]
if RUN_TRAINING:
    for architecture in ("convnext_tiny","swin_tiny"):
        result=train_candidate(architecture,resume=True)
        candidate_results.append(result)
    comparison=[]
    for result in candidate_results:
        bestrow=max(result["history"],key=lambda x:x["validation_joint_score"])
        comparison.append({"architecture":result["architecture"],"parameters":result["parameters"],"training_seconds":result["training_seconds"],"best_epoch":result["best_epoch"],"validation_macro_f1":bestrow["validation_macro_f1"],"validation_glioma_recall":bestrow["validation_glioma_recall"],"validation_balanced_accuracy":bestrow["validation_balanced_accuracy"],"validation_segmentation_dice":bestrow["validation_dice"],"validation_segmentation_iou":bestrow["validation_iou"],"joint_score":bestrow["validation_joint_score"]})
    comparison_df=pd.DataFrame(comparison).sort_values("joint_score",ascending=False)
    comparison_df.to_csv(OUTPUT_DIR/"model_comparison.csv",index=False);display(comparison_df)
else:
    print("Training disabled. Set RUN_TRAINING=True to launch the full run.")


## 12. Validation glioma threshold analysis

Normal predictions use four-class argmax. The override predicts glioma when `P(glioma) >= threshold`; otherwise it uses argmax among the remaining classes. The selected threshold is the highest swept threshold meeting the configured minimum glioma recall, then ranked by macro F1 and accuracy. This avoids selecting the lowest threshold automatically.

In [ ]:
def glioma_threshold_sweep(y,prob,target_recall=GLIOMA_RECALL_TARGET):
    rows=[]
    for t in np.arange(.05,.951,.025):
        pred=1+prob[:,1:].argmax(1); pred=np.where(prob[:,0]>=t,0,pred)
        pr,re,f1,_=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)
        rows.append({"threshold":float(t),"glioma_precision":float(pr[0]),"glioma_recall":float(re[0]),"glioma_f1":float(f1[0]),"accuracy":float(accuracy_score(y,pred)),"macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),"meningioma_recall":float(re[1]),"pituitary_recall":float(re[2]),"non_tumorous_recall":float(re[3])})
    eligible=[r for r in rows if r["glioma_recall"]>=target_recall]
    selected=max(eligible,key=lambda r:(r["threshold"],r["macro_f1"],r["accuracy"])) if eligible else max(rows,key=lambda r:(r["glioma_recall"],r["macro_f1"],r["accuracy"]))
    return pd.DataFrame(rows),selected

# Run after candidate training; chooses architecture using validation joint score only.
selected_architecture=comparison_df.iloc[0].architecture if RUN_TRAINING else None
if selected_architecture:
    ck=torch.load(CHECKPOINT_DIR/f"candidate_{selected_architecture}_best.pth",map_location=DEVICE,weights_only=False)
    selected_model=MultiTaskModel(selected_architecture,pretrained=False).to(DEVICE);selected_model.load_state_dict(ck["state_dict"])
    _,_,val_probs,_,_=validate(selected_model,val_loaders["classification"],val_loaders["segmentation"])
    val_y=np.concatenate([b.numpy() for _,b,_ in val_loaders["classification"]])
    sweep,selected_threshold=glioma_threshold_sweep(val_y,val_probs)
    sweep.to_csv(OUTPUT_DIR/"validation_glioma_threshold_sweep.csv",index=False)
    val_seg_prob=[];val_seg_truth=[]
    with torch.inference_mode():
        for vx,vy,_,_ in val_loaders["segmentation"]:
            val_seg_prob.extend(torch.sigmoid(selected_model(vx.to(DEVICE))["segmentation"]).cpu().numpy());val_seg_truth.extend(vy.numpy())
    val_seg_prob=np.asarray(val_seg_prob);val_seg_truth=np.asarray(val_seg_truth)
    mask_grid=[(seg_metrics(val_seg_prob,val_seg_truth,float(t))["dice_mean"],float(t)) for t in MASK_THRESHOLD_GRID]
    selected_mask_threshold=max(mask_grid,key=lambda z:(z[0],-abs(z[1]-.5)))[1]
    selected_threshold["selected_mask_threshold"]=selected_mask_threshold
    (OUTPUT_DIR/"selected_threshold.json").write_text(json.dumps(selected_threshold,indent=2))
    selected_threshold["target_recall_met"]=selected_threshold["glioma_recall"]>=GLIOMA_RECALL_TARGET
    selected_threshold["selection_rule"]="among target-recall candidates, maximize threshold; if none meet target, maximize recall then macro F1 and accuracy"
    print("Selected validation thresholds:",selected_threshold)


## 13. Final model selection and checkpoint export

The selected candidate is chosen by validation joint score. Its threshold is calibrated only on validation predictions. The existing EfficientNet-B0 baseline checkpoint is not opened for writing or replaced.

In [ ]:
if RUN_TRAINING and selected_architecture:
    selected_path=CHECKPOINT_DIR/f"candidate_{selected_architecture}_best.pth"
    selected_ck=torch.load(selected_path,map_location="cpu",weights_only=False)
    selected_ck["metadata"]["glioma_threshold"]=selected_threshold["threshold"]
    selected_ck["metadata"]["threshold_selection_criterion"]={"target_glioma_recall":GLIOMA_RECALL_TARGET,"selection":"highest threshold meeting target validation glioma recall; tie-break macro F1 then accuracy","selected_validation_metrics":selected_threshold}
    selected_ck["metadata"]["preprocessing"]={"input_size":IMAGE_SIZE,"resize":"PIL bilinear square resize","channels":"RGB replicated from MRI grayscale","mean":MEAN,"std":STD,"mask_resize":"nearest","mask_binary_threshold":127}
    selected_ck["metadata"]["segmentation_output"]="single-channel binary tumor logit"
    selected_ck["metadata"]["training_config"]={"epochs":EPOCHS,"batch_size":BATCH_SIZE,"gradient_accumulation":GRAD_ACCUM_STEPS,"learning_rate":LEARNING_RATE,"weight_decay":WEIGHT_DECAY,"lambda_classification":LAMBDA_CLASSIFICATION,"lambda_segmentation":LAMBDA_SEGMENTATION,"seed":SEED}
    torch.save(selected_ck,CHECKPOINT_DIR/"best_multitask_model.pth")
    (CHECKPOINT_DIR/"best_multitask_model.json").write_text(json.dumps(selected_ck["metadata"],indent=2))


## 14–16. Official test evaluation and actual visualizations

This section runs only after both candidates, the selected architecture, and the glioma threshold are frozen. It evaluates normal argmax and validation-threshold predictions separately. Segmentation metrics use the validation-selected mask threshold (selected by mean per-image Dice); absent negative masks are stated explicitly.

In [ ]:
def official_test_rows():
    cls=[]
    for folder in sorted((DATA_ROOT/"classification_task"/"test").iterdir()):
        if folder.is_dir(): cls += [{"path":str(p),"label":FOLDER_LABELS[folder.name]} for p in sorted(folder.glob("*.jpg"))]
    segbase=DATA_ROOT/"segmentation_task"/"test";imgs={p.stem:p for p in (segbase/"images").glob("*.jpg")};masks={p.stem:p for p in (segbase/"masks").glob("*.png")}
    if imgs.keys()!=masks.keys(): raise ValueError("Official test image-mask pairing mismatch")
    seg=[{"image":str(imgs[k]),"mask":str(masks[k]),"label":mask_class_from_name(imgs[k])} for k in sorted(imgs)]
    return cls,seg

if RUN_TRAINING and selected_architecture:
    test_cls,test_seg=official_test_rows()
    tlc=DataLoader(MRIRecordDataset(test_cls,"classification"),batch_size=BATCH_SIZE,num_workers=NUM_WORKERS)
    tls=DataLoader(MRIRecordDataset(test_seg,"segmentation"),batch_size=BATCH_SIZE,num_workers=NUM_WORKERS)
    test_metrics,test_y,test_prob,test_seg_prob,test_seg_truth=validate(selected_model,tlc,tls)
    argmax_pred=test_prob.argmax(1);threshold_pred=1+test_prob[:,1:].argmax(1);threshold_pred=np.where(test_prob[:,0]>=selected_threshold["threshold"],0,threshold_pred)
    test_metrics["classification_argmax"]=cls_metrics(test_y,test_prob)
    # Threshold metrics are evaluated without changing the probability estimates.
    pr,re,f1,_=precision_recall_fscore_support(test_y,threshold_pred,labels=range(4),zero_division=0)
    test_metrics["classification_glioma_threshold_override"]={"threshold":selected_threshold["threshold"],"accuracy":float(accuracy_score(test_y,threshold_pred)),"macro_f1":float(f1_score(test_y,threshold_pred,average="macro",zero_division=0)),"per_class_precision":pr.tolist(),"per_class_recall":re.tolist(),"per_class_f1":f1.tolist(),"confusion_matrix":confusion_matrix(test_y,threshold_pred,labels=range(4)).tolist(),"threshold_source":"validation only"}
    # Validation mask threshold sweep is selected by per-image mean Dice.
    # Official test is NOT used to select a threshold: use validation grid optimum.
    val_sp=[]
    with torch.inference_mode():
        for x,y,_,_ in val_loaders["segmentation"]: val_sp.extend(torch.sigmoid(selected_model(x.to(DEVICE))["segmentation"]).cpu().numpy())
    val_sy=np.concatenate([y.numpy() for _,y,_,_ in val_loaders["segmentation"]])
    val_candidates=[(seg_metrics(np.asarray(val_sp),val_sy,float(t))["dice_mean"],float(t)) for t in MASK_THRESHOLD_GRID]
    selected_mask_threshold=max(val_candidates,key=lambda z:(z[0],-abs(z[1]-.5)))[1]
    test_metrics["segmentation"] = seg_metrics(test_seg_prob,test_seg_truth,selected_mask_threshold)
    test_metrics["segmentation"]["selected_mask_threshold"]=selected_mask_threshold
    test_metrics["segmentation"]["threshold_source"]="validation"
    test_metrics["threshold_selection"]={"glioma":selected_threshold,"mask_threshold":selected_mask_threshold,"negative_segmentation_masks_present":DATA_AUDIT["negative_masks"]}
    (OUTPUT_DIR/"test_metrics.json").write_text(json.dumps(test_metrics,indent=2,default=lambda o:o.tolist() if hasattr(o,"tolist") else str(o)))
    (OUTPUT_DIR/"validation_metrics.json").write_text(json.dumps(ck["metadata"]["validation_metrics"],indent=2,default=lambda o:o.tolist() if hasattr(o,"tolist") else str(o)))
    print(json.dumps(test_metrics,indent=2,default=lambda o:o.tolist() if hasattr(o,"tolist") else str(o)))


## 17. Classification plots, ROC, Grad-CAM, and segmentation examples

In [ ]:
def save_confusion_roc(y,prob,pred,prefix):
    cm=confusion_matrix(y,pred,labels=range(4));fig,ax=plt.subplots(figsize=(6,5));im=ax.imshow(cm,cmap="Blues");fig.colorbar(im,ax=ax)
    ax.set_xticks(range(4),CLASSES,rotation=30,ha="right");ax.set_yticks(range(4),CLASSES);ax.set(xlabel="Predicted",ylabel="True",title=f"{prefix} confusion matrix")
    for i in range(4):
        for j in range(4):ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    fig.tight_layout();fig.savefig(PLOT_DIR/f"{prefix}_confusion_matrix.png",dpi=160);plt.close(fig)
    ybin=label_binarize(y,classes=range(4));fig,ax=plt.subplots(figsize=(6,5))
    for i,c in enumerate(CLASSES):
        if len(np.unique(ybin[:,i]))<2:continue
        fpr,tpr,_=roc_curve(ybin[:,i],prob[:,i]);ax.plot(fpr,tpr,label=f"{c} AUC={auc(fpr,tpr):.3f}")
    ax.plot([0,1],[0,1],"k--");ax.legend();ax.set(xlabel="False positive rate",ylabel="True positive rate",title=f"{prefix} one-vs-rest ROC");fig.tight_layout();fig.savefig(PLOT_DIR/f"{prefix}_roc.png",dpi=160);plt.close(fig)

def gradcam(model,image_tensor,class_index=None):
    captured={}
    def hook(_m,_i,out):out.retain_grad();captured["activation"]=out
    handle=model.encoder[7].register_forward_hook(hook);model.zero_grad(set_to_none=True)
    logits=model(image_tensor)["classification"];k=int(logits.argmax(1)[0]) if class_index is None else int(class_index);logits[0,k].backward();act=captured["activation"];grad=act.grad
    if model.architecture=="swin_tiny":act=act.permute(0,3,1,2);grad=grad.permute(0,3,1,2)
    weights=grad.mean((2,3),keepdim=True);cam=torch.relu((weights*act).sum(1,keepdim=True));cam=F.interpolate(cam,size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)[0,0].detach().cpu().numpy();handle.remove();model.zero_grad(set_to_none=True)
    return cam/(cam.max()+1e-8)

if RUN_TRAINING and selected_architecture:
    save_confusion_roc(test_y,test_prob,argmax_pred,"test_argmax")
    save_confusion_roc(test_y,test_prob,threshold_pred,"test_glioma_threshold")
    fig,ax=plt.subplots(1,2,figsize=(9,4));sm=seg_metrics(test_seg_prob,test_seg_truth,selected_mask_threshold);ax[0].hist(sm["dice_per_image"],bins=20,range=(0,1));ax[0].set_title("Test per-image Dice");ax[1].hist(sm["iou_per_image"],bins=20,range=(0,1));ax[1].set_title("Test per-image IoU");fig.tight_layout();fig.savefig(PLOT_DIR/"test_dice_iou.png",dpi=160);plt.close(fig)
    # Actual selected model examples; category names are computed from prediction/ground truth.
    all_prob=[];all_masks=[];all_targets=[];all_inputs=[]
    with torch.inference_mode():
        for x,y,_,_ in tls:
            o=selected_model(x.to(DEVICE));all_inputs.extend(x);all_targets.extend(y.numpy());all_masks.extend(torch.sigmoid(o["segmentation"]).cpu().numpy())
    ds=np.asarray(all_masks);gt=np.asarray(all_targets);dicev=seg_metrics(ds,gt,selected_mask_threshold)["dice_per_image"]
    fpcount=((ds>=selected_mask_threshold)&(gt<.5)).sum((1,2,3));fncount=((ds<selected_mask_threshold)&(gt>=.5)).sum((1,2,3))
    chosen={"correct":int(np.argmax(dicev)),"poor":int(np.argmin(dicev))}
    if int(np.max(fpcount))>0: chosen["false_positive_pixels"]=int(np.argmax(fpcount))
    if int(np.max(fncount))>0: chosen["false_negative_pixels"]=int(np.argmax(fncount))
    fig,axes=plt.subplots(len(chosen),4,figsize=(12,3*len(chosen)),squeeze=False)
    for row,(name,idx) in enumerate(chosen.items()):
        original=(all_inputs[idx].permute(1,2,0).numpy()*np.array(STD)+np.array(MEAN)).clip(0,1)
        axes[row,0].imshow(original);axes[row,0].set_title(f"MRI: {name}")
        axes[row,1].imshow(gt[idx,0],cmap="gray");axes[row,1].set_title("Ground truth")
        axes[row,2].imshow(ds[idx,0]>=selected_mask_threshold,cmap="gray");axes[row,2].set_title("Predicted mask")
        axes[row,3].imshow(original);axes[row,3].imshow(ds[idx,0]>=selected_mask_threshold,alpha=.35,cmap="Reds");axes[row,3].set_title(f"Overlay; Dice={dicev[idx]:.3f}")
        for ax in axes[row]:ax.axis("off")
    fig.tight_layout();fig.savefig(PLOT_DIR/"test_segmentation_examples.png",dpi=160);plt.close(fig)
    if "false_positive_pixels" not in chosen: print("No false-positive pixel example in these predictions; none will be fabricated.")
    if "false_negative_pixels" not in chosen: print("No false-negative pixel example in these predictions; none will be fabricated.")
    # Grad-CAM explanation on one real test image, distinct from segmentation.
    sample=torch.from_numpy(np.asarray(ImageOps.exif_transpose(Image.open(test_cls[0]["path"])).convert("RGB").resize((IMAGE_SIZE,IMAGE_SIZE)),dtype=np.float32).copy()).permute(2,0,1)/255
    sample=(sample-torch.tensor(MEAN)[:,None,None])/torch.tensor(STD)[:,None,None];selected_model.eval();cam=gradcam(selected_model,sample.unsqueeze(0).to(DEVICE))
    plt.imsave(PLOT_DIR/"test_classifier_gradcam.png",cam,cmap="magma")


## 18–19. Checkpoint export and inference smoke test

The final checkpoint is the selected shared model. This cell saves reproducibility metadata and does a shape/inference check. The application loads only `best_multitask_model.pth`; it never trains.

In [ ]:
if RUN_TRAINING and selected_architecture:
    final_meta={"architecture":selected_architecture,"classes":CLASSES,"input_size":IMAGE_SIZE,"normalization":{"mean":MEAN,"std":STD},"grayscale_to_rgb":True,"glioma_threshold":selected_threshold["threshold"],"threshold_selection_criterion":selected_threshold,"mask_threshold":selected_mask_threshold,"segmentation_output":"binary single-channel logits","task_loss_weights":{"classification":LAMBDA_CLASSIFICATION,"segmentation":LAMBDA_SEGMENTATION},"seed":SEED,"dataset_root":"data/raw/brisc2025","split_references":[f"data/splits/{f}" for f in ("classification_train.csv","classification_validation.csv","segmentation_train.csv","segmentation_validation.csv")],"validation_metrics":ck["metadata"]["validation_metrics"],"test_metrics_reference":"outputs/test_metrics.json","preprocessing":{"resize":"PIL bilinear square resize","normalization_mean":MEAN,"normalization_std":STD,"mask_resize":"nearest","mask_binary_threshold":127},"pretrained_weights":"ConvNeXt_Tiny_Weights.DEFAULT" if selected_architecture=="convnext_tiny" else "Swin_T_Weights.DEFAULT","torch":str(torch.__version__),"torchvision":str(torchvision.__version__)}
    torch.save({"state_dict":selected_model.state_dict(),"metadata":final_meta},CHECKPOINT_DIR/"best_multitask_model.pth")
    (CHECKPOINT_DIR/"best_multitask_model.json").write_text(json.dumps(final_meta,indent=2))
    # Trainer state remains resumable; latest_checkpoint.pth points to the selected candidate's full optimizer state.
    shutil.copy2(CHECKPOINT_DIR/f"latest_{selected_architecture}.pth",CHECKPOINT_DIR/"latest_checkpoint.pth")
    (REPORT_DIR/"run_summary.json").write_text(json.dumps({"selected_architecture":selected_architecture,"validation":final_meta["validation_metrics"],"test":test_metrics,"model_comparison":"outputs/model_comparison.csv"},indent=2,default=lambda o:o.tolist() if hasattr(o,"tolist") else str(o)))
    with torch.inference_mode(): out=selected_model(torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE))
    assert out["classification"].shape==(1,4) and out["segmentation"].shape==(1,1,IMAGE_SIZE,IMAGE_SIZE)
    print("Checkpoint exported and inference shape check passed.")
else:
    print("Training/export cells are ready; no checkpoint or metrics were created in this setup-only session.")


## Colab and local execution

1. Install compatible PyTorch/torchvision from the official PyTorch selector; install the packages in `requirements.txt`.
2. In Colab, mount Drive or upload the dataset and set `DATA_ROOT`, `SPLIT_DIR`, and `OUTPUT_ROOT` in the configuration cell.
3. Run cells through the audits first. Confirm the printed path/counts.
4. Run candidates; latest checkpoints resume after a disconnect by rerunning with the same architecture and `resume=True`.
5. Run validation calibration/model selection, then official test, then export.
6. Locally launch `streamlit run app.py`.

The BRISC `non_tumorous` class is not guaranteed healthy. This is slice-level research software, not clinical diagnosis. No patient identifiers were available, so subject-level independence is not guaranteed.